## PARTIE 2 : Deep Learning - Classification d'images

In [ ]:
import numpy as np
import pandas as pd

fichier = "WangSignatures.xlsx"

noms_onglets = {
    "JCD": "WangSignaturesJCD",
    "PHOG": "WangSignaturesPHOG",
    "CEDD": "WangSignaturesCEDD",
    "FCTH": "WangSignaturesFCTH",
    "FuzzyColor": "WangSignaturesFuzzyColorHistogr"
}

# Chargement des cinq descripteurs
descripteurs = {}

for nom, onglet in noms_onglets.items():
    df = pd.read_excel(
        fichier,
        sheet_name=onglet,
        header=None,
        index_col=0
    )

    descripteurs[nom] = df

    print(f"{nom} : {df.shape}")

# Vérification des noms d'images
ordre_images = descripteurs["JCD"].index

assert len(ordre_images) == 1000
assert ordre_images.is_unique

for nom, df in descripteurs.items():
    assert df.index.is_unique
    assert len(df) == 1000
    assert set(df.index) == set(ordre_images)

    descripteurs[nom] = df.loc[ordre_images]

# Construction des labels
labels = np.array([
    int(nom[:-4]) // 100
    for nom in ordre_images
])

print("\nRépartition des classes :")
classes, effectifs = np.unique(labels, return_counts=True)

for classe, effectif in zip(classes, effectifs):
    print(f"Classe {classe} : {effectif} images")

# Conversion en matrices NumPy
X_descripteurs = {}

for nom, df in descripteurs.items():
    X_descripteurs[nom] = df.to_numpy(dtype=float)

# Concaténation des cinq descripteurs
X_complet = np.concatenate(
    list(X_descripteurs.values()),
    axis=1
)

# Vérifications finales
assert X_complet.shape[0] == len(labels)
assert np.isfinite(X_complet).all()

print("\nDimensions des matrices :")

for nom, X in X_descripteurs.items():
    print(f"{nom} : {X.shape}")

print(f"\nTous les descripteurs : {X_complet.shape}")
print(f"Labels : {labels.shape}")

In [ ]:
from sklearn.model_selection import train_test_split

indices = np.arange(len(labels))

idx_train_val, idx_test = train_test_split(
    indices,
    test_size=0.2,
    stratify=labels,
    random_state=42
)

idx_train, idx_val = train_test_split(
    idx_train_val,
    test_size=0.2,
    stratify=labels[idx_train_val],
    random_state=42
)

y_train = labels[idx_train]
y_val = labels[idx_val]
y_test = labels[idx_test]

print("Entraînement :", len(idx_train))
print("Validation :", len(idx_val))
print("Test :", len(idx_test))

In [ ]:
from sklearn.preprocessing import StandardScaler

X = X_descripteurs["JCD"]

X_train = X[idx_train]
X_val = X[idx_val]
X_test = X[idx_test]

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

print(X_train.shape)
print(X_val.shape)
print(X_test.shape)

In [ ]:
from tensorflow import keras
from tensorflow.keras import layers

def creer_modele(nb_entrees):
    model = keras.Sequential([
        keras.Input(shape=(nb_entrees,)),

        layers.Dense(64, activation="relu"),
        layers.Dense(32, activation="relu"),

        layers.Dense(10, activation="softmax")
    ])

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.001),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model

model = creer_modele(X_train.shape[1])

model.summary()

In [ ]:
history = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=100,
    batch_size=32,
    verbose=1
)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)

plt.plot(history.history["loss"], label="Train")
plt.plot(history.history["val_loss"], label="Validation")

plt.xlabel("Époque")
plt.ylabel("Loss")
plt.title("Évolution de la loss")
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)

plt.plot(history.history["accuracy"], label="Train")
plt.plot(history.history["val_accuracy"], label="Validation")

plt.xlabel("Époque")
plt.ylabel("Accuracy")
plt.title("Évolution de l'accuracy")
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    ConfusionMatrixDisplay
)

probas = model.predict(X_val, verbose=0)
predictions = np.argmax(probas, axis=1)

accuracy = accuracy_score(y_val, predictions)
taux_erreur = 1 - accuracy

print(f"Accuracy : {accuracy:.4f}")
print(f"Taux d'erreur : {taux_erreur:.4f}")

matrice = confusion_matrix(y_val, predictions)

disp = ConfusionMatrixDisplay(
    confusion_matrix=matrice,
    display_labels=np.arange(10)
)

disp.plot(cmap="Blues")
plt.title("Matrice de confusion - JCD")
plt.show()

In [ ]:
resultats = {}

ensembles = {
    **X_descripteurs,
    "Complet": X_complet
}

for nom, X in ensembles.items():

    print(f"\n===== {nom} =====")

    X_train = X[idx_train]
    X_val = X[idx_val]

    scaler = StandardScaler()

    X_train = scaler.fit_transform(X_train)
    X_val = scaler.transform(X_val)

    keras.utils.set_random_seed(42)
    model = creer_modele(X_train.shape[1])

    early_stopping = keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=10,
        restore_best_weights=True
    )

    history = model.fit(
        X_train,
        y_train,
        validation_data=(X_val, y_val),
        epochs=100,
        batch_size=32,
        callbacks=[early_stopping],
        verbose=0
    )

    probas = model.predict(X_val, verbose=0)
    predictions = np.argmax(probas, axis=1)

    accuracy = accuracy_score(y_val, predictions)

    resultats[nom] = {
        "accuracy": accuracy,
        "taux_erreur": 1 - accuracy,
        "epochs": len(history.history["loss"]),
        "model": model,
        "scaler": scaler,
        "history": history,
        "predictions": predictions
    }

    print(f"Accuracy validation : {accuracy:.4f}")
    print(f"Taux d'erreur : {1 - accuracy:.4f}")
    print(f"Époques : {len(history.history['loss'])}")

In [ ]:
tableau = pd.DataFrame({
    nom: {
        "Accuracy": r["accuracy"],
        "Taux d'erreur": r["taux_erreur"],
        "Époques": r["epochs"]
    }
    for nom, r in resultats.items()
}).T

tableau = tableau.sort_values(
    by="Accuracy",
    ascending=False
)

display(tableau)

In [ ]:
plt.figure(figsize=(9, 5))

plt.bar(
    tableau.index,
    tableau["Accuracy"]
)

plt.xlabel("Descripteur")
plt.ylabel("Accuracy validation")
plt.title("Comparaison des descripteurs")
plt.ylim(0, 1)
plt.grid(axis="y")

plt.show()

In [ ]:
meilleur_nom = tableau.index[0]

meilleur = resultats[meilleur_nom]

print("Meilleur descripteur :", meilleur_nom)

ConfusionMatrixDisplay.from_predictions(
    y_val,
    meilleur["predictions"],
    labels=np.arange(10),
    cmap="Blues"
)

plt.title(f"Matrice de confusion - {meilleur_nom}")
plt.show()